# 01 - Data Exploration

This notebook inspects the six fruit freshness classes before training. Run it after creating the directory split described in `dataset/README.md`.

## 1. Import Libraries

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

## 2. Discover the Dataset

In [ ]:
DATASET_DIRECTORY = '../dataset'
TRAIN_DIRECTORY = os.path.join(DATASET_DIRECTORY, 'train')
CLASS_NAMES = [
    'fresh_apple', 'rotten_apple',
    'fresh_banana', 'rotten_banana',
    'fresh_orange', 'rotten_orange',
]
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

if not os.path.isdir(TRAIN_DIRECTORY):
    raise FileNotFoundError('Prepare dataset/train before running this notebook.')

image_paths = {
    name: sorted(
        os.path.join(TRAIN_DIRECTORY, name, filename)
        for filename in os.listdir(os.path.join(TRAIN_DIRECTORY, name))
        if os.path.isfile(os.path.join(TRAIN_DIRECTORY, name, filename))
        and os.path.splitext(filename)[1].lower() in IMAGE_EXTENSIONS
    )
    for name in CLASS_NAMES
}

## 3. Image Counts and Class Distribution

Large differences between class counts can bias a classifier, so the distribution should be inspected before training.

In [ ]:
class_counts = {name: len(paths) for name, paths in image_paths.items()}
for class_name, count in class_counts.items():
    print(f'{class_name:15s}: {count:5d}')

plt.figure(figsize=(10, 5))
plt.bar(CLASS_NAMES, [class_counts[name] for name in CLASS_NAMES], color='#4f7c5d')
plt.title('Training Image Distribution')
plt.xlabel('Class')
plt.ylabel('Number of Images')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

## 4. Sample Images from Every Class

In [ ]:
figure, axes = plt.subplots(2, 3, figsize=(12, 7))
for axis, class_name in zip(axes.flat, CLASS_NAMES):
    if not image_paths[class_name]:
        axis.text(0.5, 0.5, 'No images', ha='center', va='center')
    else:
        bgr_image = cv2.imread(str(image_paths[class_name][0]))
        if bgr_image is None:
            axis.text(0.5, 0.5, 'Unreadable image', ha='center', va='center')
        else:
            rgb_image = cv2.cvtColor(bgr_image, cv2.COLOR_BGR2RGB)
            axis.imshow(rgb_image)
    axis.set_title(class_name.replace('_', ' ').title())
    axis.axis('off')
plt.tight_layout()
plt.show()

## 5. Inspect Original Image Dimensions

Photographs can have different shapes. The CNN pipeline later standardizes all of them to 150 × 150 RGB pixels.

In [ ]:
dimensions = []
for paths in image_paths.values():
    for path in paths:
        image = cv2.imread(str(path))
        if image is not None:
            height, width = image.shape[:2]
            dimensions.append((width, height))

dimensions = np.asarray(dimensions)
if dimensions.size:
    print('Images inspected:', len(dimensions))
    print('Minimum (width, height):', dimensions.min(axis=0))
    print('Maximum (width, height):', dimensions.max(axis=0))
    print('Mean (width, height):', dimensions.mean(axis=0).round(1))

    plt.figure(figsize=(7, 5))
    plt.scatter(dimensions[:, 0], dimensions[:, 1], alpha=0.35)
    plt.axvline(150, color='darkred', linestyle='--', label='CNN width')
    plt.axhline(150, color='darkgreen', linestyle='--', label='CNN height')
    plt.xlabel('Original width')
    plt.ylabel('Original height')
    plt.title('Original Image Dimensions')
    plt.legend()
    plt.show()

## Observations

After running the cells, record genuine observations about class balance, image quality, lighting, backgrounds, and dimensions here. Do not infer freshness beyond visible surface appearance.